In [1]:
!pip install -qU langchain
!pip install -qU langchain-google-vertexai
!pip install -qU langchain-huggingface
!pip install -qU langchain-qdrant
!pip install -qU langchain-community
!pip install -qU langgraph
!pip install fastembed
!pip install datasets
!pip install sqlalchemy
!pip install emoji

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 437.6/437.6 kB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.4/99.4 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 MB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 68.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9

In [4]:
import os
from google.colab import userdata

# os.environ["GOOGLE_API_KEY"] = userdata.get('gemini_api_key')
os.environ["LANGSMITH_API_KEY"] = userdata.get('LANGSMITH_API_KEY')
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["QDRANT_API_KEY"] = userdata.get('QDRANT_API_KEY')
os.environ["DATABASE_URL"] = userdata.get('DATABASE_URL')

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_qdrant import FastEmbedSparse, QdrantVectorStore, RetrievalMode
from qdrant_client import QdrantClient, models
from qdrant_client.http.models import Distance, SparseVectorParams, VectorParams

model_kwargs = {'trust_remote_code': True}
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/LaBSE",model_kwargs=model_kwargs)
model = HuggingFaceCrossEncoder(model_name="Alibaba-NLP/gte-reranker-modernbert-base")
sparse_embeddings = FastEmbedSparse(model_name="Qdrant/bm25")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/461 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/2.02k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

swedish.txt:   0%|          | 0.00/559 [00:00<?, ?B/s]

slovene.txt:   0%|          | 0.00/16.0k [00:00<?, ?B/s]

tajik.txt:   0%|          | 0.00/1.82k [00:00<?, ?B/s]

romanian.txt:   0%|          | 0.00/1.91k [00:00<?, ?B/s]

turkish.txt:   0%|          | 0.00/260 [00:00<?, ?B/s]

In [7]:
# Set your Qdrant endpoint and API key
client = QdrantClient(
    url="https://ff9ebd5e-31f8-4e7a-a866-ef23b9b9cd5e.us-west-2-0.aws.cloud.qdrant.io",
    api_key=os.environ["QDRANT_API_KEY"],
)

if not client.collection_exists(collection_name="cp-genie"):
  client.recreate_collection(
      collection_name="cp-genie",
      vectors_config={"dense": VectorParams(size=768, distance=Distance.COSINE)},
      sparse_vectors_config={
          "sparse": SparseVectorParams(index=models.SparseIndexParams(on_disk=False))
      },
  )

<ipython-input-7-9bfba98708f2>:7: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


True

In [11]:
# models.py
from sqlalchemy import Column, Integer, String, DateTime, ForeignKey
from sqlalchemy.ext.declarative import declarative_base

Base = declarative_base()

class Metadata(Base):
    __tablename__ = 'metadata'

    source_url = Column(String, primary_key=True)
    content_type = Column(String)
    # using last_mod as string
    last_modified = Column(String)
    fetch_timestamp = Column(DateTime)
    raw_filepath = Column(String)


class Content(Base):
    __tablename__ = 'content'

    id = Column(Integer, primary_key=True, autoincrement=True)
    source_path = Column(String)
    content = Column(String)


class Embedded(Base):
    __tablename__ = 'embedded'

    id = Column(Integer, autoincrement=True, primary_key=True)
    content_id = Column(Integer, ForeignKey('content.id'))
    cleaned_content = Column(String)
    last_updated = Column(DateTime)

<ipython-input-11-a0bdacf7addb>:5: MovedIn20Warning: The ``declarative_base()`` function is now available as sqlalchemy.orm.declarative_base(). (deprecated since: 2.0) (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  Base = declarative_base()


In [16]:
import sqlalchemy
from sqlalchemy import create_engine, text
from sqlalchemy.orm import sessionmaker
from datetime import datetime

# Replace with your Neon database connection string
engine = create_engine(os.environ["DATABASE_URL"])
Session = sessionmaker(bind=engine)
session = Session()

# Create table postgreSQL if not exist
Base.metadata.create_all(engine)

In [17]:
def retrieve_text_content():
    """Retrieves text content from the specified table in Neon."""
    try:
        existing_content = {
          c.id: c.content
          for c in session.query(Content.id, Content.content).all()
        }

        return existing_content

    except sqlalchemy.exc.SQLAlchemyError as e:
        session.rollback()
        print(f"An error occurred: {e}")
        return []

def retrieve_embedded_content():
    """Retrieves embedded id that already embedded from the specified table in Neon."""
    try:
      results = session.query(Embedded.content_id, Embedded.last_updated).all()
      embedded_content_id = {
        r.content_id : r.last_updated
        for r in results
      }

      return embedded_content_id

    except sqlalchemy.exc.SQLAlchemyError as e:
        session.rollback()
        print(f"An error occurred: {e}")

        return {}

text_content_list = retrieve_text_content()
old_embedded_content_id = retrieve_embedded_content()

In [18]:
import re
from tqdm import tqdm
import emoji

def clean_text(text):
    # Remove unwanted characters and whitespace
    cleaned_text = emoji.replace_emoji(text, '')
    cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()

    return cleaned_text

In [24]:
docs = []
new_id_docs = {}

for id, text in tqdm(text_content_list.items(), desc="Retrieving the new content", position=0, leave=True):
    if id in old_embedded_content_id.keys():
        print(f"[SKIPPED] Content {id} already in QDrant")
        continue

    cleaned = clean_text(text)
    docs.append(cleaned)
    new_id_docs[id] = cleaned

docs[:5]

Retrieving the new content: 100%|██████████| 516/516 [00:00<00:00, 794.14it/s] 


['ประมวลภาพกิจกรรมปฐมนิเทศฝึกงานของนิสิต CEDT ชั้นปีที่ 1 และชั้นปีที่2 Tweet May 4, 2025 กิจกรรมนิสิต , ปริญญาตรี Tag: CEDT , CP , CPCU , CUCA , วิศวะคอมจุฬา ประมวลภาพกิจกรรมปฐมนิเทศฝึกงานของนิสิต CEDT ชั้นปีที่ 1 และชั้นปีที่2 ภาควิชาวิศวกรรมคอมพิวเตอร์ คณะวิศวกรรมศาสตร์ จุฬาลงกรณ์มหาวิทยาลัย ได้จัดกิจกรรมปฐมนิเทศฝึกงานสำหรับนิสิตหลักสูตรวิศวกรรมคอมพิวเตอร์และเทคโนโลยีดิจิทัล (CEDT) ชั้นปีที่ 1 และชั้นปีที่ 2 อย่างยิ่งใหญ่ ณ หอประชุมคณะวิศวกรรมศาสตร์ และอาคารจุฬาพัฒน์ 3 เพื่อเตรียมความพร้อมก่อนก้าวเข้าสู่สนามจริงในสถานประกอบการกว่า 120 แห่งทั่วประเทศ กิจกรรมครั้งนี้ไม่เพียงเป็นการให้ข้อมูลเบื้องต้นเกี่ยวกับการฝึกงานเท่านั้น แต่ยังเป็นเวทีสำคัญที่นิสิตได้เปิดโลกทัศน์ เรียนรู้แนวทางการทำงานในสายวิศวกรรมคอมพิวเตอร์จากคณาจารย์ผู้ทรงคุณวุฒิของหลักสูตร โดย ผศ.ดร.เอกพล ช่วงสุวนิช ได้ให้คำแนะนำอย่างเจาะลึกเกี่ยวกับสิ่งที่นิสิตควรรู้และควรเตรียมตัวก่อนเข้าสู่สถานประกอบการ ขณะที่ ผศ.ดร.ณรงค์เดช กีรติพรานนท์ ได้อธิบายระบบการดูแลนิสิตฝึกงานอย่างเป็นระบบ พร้อมแนวทางการประเมินผลที่โปร่งใสและสร้างส

In [27]:
from langchain.schema import Document
# Turn to Document

def document_transformer(text: str) -> Document:
    return Document(page_content=text)

In [28]:
qdocs = [document_transformer(doc) for doc in docs]

In [29]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,      # number of characters
    chunk_overlap=100,    # to retain context between chunks
    separators=["\n\n", "\n", ".", " "],
)

split_docs = text_splitter.split_documents(qdocs)

In [32]:
qdrant = QdrantVectorStore(
    client=client,
    collection_name="cp-genie",
    embedding=embeddings,
    sparse_embedding=sparse_embeddings,
    retrieval_mode=RetrievalMode.HYBRID,
    vector_name="dense",
    sparse_vector_name="sparse",
)


try:
  qdrant.add_documents(documents=split_docs)

  BATCH_SIZE = 20
  count = 0
  for id, text in new_id_docs.items():
      record = Embedded(
          content_id= id,
          cleaned_content=text,
          last_updated=datetime.now()
      )

      session.merge(record)

      count += 1
      if count % BATCH_SIZE == 0:
          session.commit()
          print(f"[OK] Sent {count} data to DB")
          count = 0

  session.commit()
  print(f"[OK] Sent {count} data to DB")
except Exception as e:
  print(f"[ERROR] Error occured while sending to QDrant: {e}")
  session.rollback()
finally:
  session.close()

[OK] Sent 20 data to DB
[OK] Sent 20 data to DB
[OK] Sent 20 data to DB
[OK] Sent 20 data to DB
[OK] Sent 20 data to DB
[OK] Sent 20 data to DB
[OK] Sent 20 data to DB
[OK] Sent 20 data to DB
[OK] Sent 20 data to DB
[OK] Sent 20 data to DB
[OK] Sent 20 data to DB
[OK] Sent 20 data to DB
[OK] Sent 20 data to DB
[OK] Sent 20 data to DB
[OK] Sent 20 data to DB
[OK] Sent 20 data to DB
[OK] Sent 20 data to DB
[OK] Sent 20 data to DB
[OK] Sent 20 data to DB
[OK] Sent 20 data to DB
[OK] Sent 20 data to DB
[OK] Sent 20 data to DB
[OK] Sent 20 data to DB
[OK] Sent 20 data to DB
[OK] Sent 20 data to DB
[OK] Sent 16 data to DB
